# Acto 2 - Separar lo real de lo falso
## Que senales puede controlar el creador de contenido antes de publicar?

**Asignatura:** SCY1101 - Programacion para la Ciencia de Datos  
**Integrante:** Arelis  
**Dataset:** Social Media Engagement Dataset - 12.000 posts x 28 columnas x 5 plataformas

---

### Narrativa

> *"Nosotros queremos darle al creador de contenido senales que pueda controlar antes de publicar."*

Adolfo clasifico el tono de cada post desde el texto con F1=0.937. Este acto parte desde ahi y agrega una capa: ese tono es **coherente** con la agresividad con que esta escrito?

Antes de responder esa pregunta, hay que limpiar el terreno: el `engagement_rate` no puede ser nuestro target porque sus componentes (likes, shares, comments) generan **data leakage**, y ademas encontramos posts con metricas matematicamente imposibles.

### Flujo completo

```
PASO 1 -> Regla matematica      Detectar fake engagement (rate > 1)
PASO 2 -> Variables sin leakage Construir coherencia_sentimiento
PASO 3 -> PCA                   Tienen estructura las senales de contexto?
PASO 4 -> K-Means               Existen grupos naturales?
PASO 5 -> Trade-off timing      Cuando publicar? Impresiones vs Engagement
PASO 6 -> Clasificacion         Puede predecirse la coherencia?
PASO 7 -> Regresion             Por que el engagement no es predecible?
PASO 8 -> Optimizacion          GridSearchCV + RandomizedSearchCV
```


---
## Setup

In [ ]:
import warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, f1_score,
                              classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)
from sklearn.model_selection import (StratifiedKFold, KFold,
                                     cross_val_score, cross_validate,
                                     GridSearchCV, RandomizedSearchCV,
                                     train_test_split)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (GradientBoostingClassifier,
                               RandomForestClassifier, RandomForestRegressor)
from sklearn.svm import SVC
from scipy.stats import randint

SEED = 42
np.random.seed(SEED)

# Estilo visual
plt.rcParams['figure.facecolor'] = '#0D1117'
plt.rcParams['axes.facecolor']   = '#21262D'
plt.rcParams['text.color']       = '#E6EDF3'
plt.rcParams['axes.labelcolor']  = '#8B949E'
plt.rcParams['xtick.color']      = '#8B949E'
plt.rcParams['ytick.color']      = '#8B949E'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

CORAL = '#F96167'; GREEN = '#3FB950'; BLUE  = '#58A6FF'
GOLD  = '#F9E795'; WHITE = '#E6EDF3'; MUTED = '#8B949E'

print('Setup completo | SEED =', SEED)


In [ ]:
# Carga del dataset
# En Colab subir el archivo directamente:
# from google.colab import files; uploaded = files.upload()
# df_raw = pd.read_csv(list(uploaded.keys())[0])
# O desde Drive:
# from google.colab import drive; drive.mount('/content/drive')
# df_raw = pd.read_csv('/content/drive/MyDrive/Social_Media_Engagement_Dataset.csv')

df_raw = pd.read_csv('Social_Media_Engagement_Dataset.csv')
print(f'Dataset cargado: {df_raw.shape[0]:,} posts x {df_raw.shape[1]} columnas')
df_raw.head(3)


---
## PASO 1 - Regla matematica: deteccion de Fake Engagement

### Por que el engagement_rate NO es nuestro target?

**Razon 1 - Data leakage:**
```
engagement_rate = (likes + shares + comments) / impressions
```
Si usamos likes_count, shares_count o comments_count como features para predecir engagement_rate,
el modelo reconstruye la formula matematicamente. Obtiene R2=0.99 sin aprender nada util.

**Razon 2 - Datos imposibles:**
Si engagement_rate > 1 hay mas reacciones que vistas. Matematicamente imposible de forma organica.
La deteccion no necesita modelo, es una regla matematica pura.


In [ ]:
# Construccion de variables y deteccion de fake
df = df_raw.copy()

# Variables temporales
df['timestamp']       = pd.to_datetime(df['timestamp'])
df['hora']            = df['timestamp'].dt.hour
df['dia_semana']      = df['timestamp'].dt.day_name()
df['mes']             = df['timestamp'].dt.month_name()
df['es_fin_semana']   = df['day_of_week'].isin(['Saturday','Sunday']).astype(int)
df['es_hora_laboral'] = df['hora'].between(9, 18).astype(int)

# Variables de contenido
df['mentions']             = df['mentions'].fillna('none')
df['tiene_mencion']        = (df['mentions'] != 'none').astype(int)
df['n_hashtags']           = df['hashtags'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) and x != '' else 0)
df['intensidad_sentimiento'] = df['sentiment_score'].abs()

# Variable construida: coherencia del sentimiento
# Conecta con el trabajo de Adolfo (usa sentiment_label que el clasifico)
# Coherente = tono y agresividad van en la misma direccion
df['coherencia_sentimiento'] = np.where(
    ((df['sentiment_label'] == 'Positive') & (df['toxicity_score'] < 0.5)) |
    ((df['sentiment_label'] == 'Negative') & (df['toxicity_score'] > 0.3)) |
    (df['sentiment_label'] == 'Neutral'),
    1, 0  # 1 = coherente, 0 = incoherente
)

df['log_engagement'] = np.log1p(df['engagement_rate'])

# REGLA MATEMATICA: fake engagement
df['es_fake'] = (df['engagement_rate'] > 1).astype(int)

n_fake = df['es_fake'].sum()
n_org  = len(df) - n_fake

print('=' * 50)
print('DETECCION DE FAKE ENGAGEMENT')
print('=' * 50)
print(f'Total posts:     {len(df):,}')
print(f'Fake (rate > 1): {n_fake:,}  ({n_fake/len(df)*100:.1f}%)')
print(f'Organicos:       {n_org:,}  ({n_org/len(df)*100:.1f}%)')

df_limpio = df[df['es_fake'] == 0].copy()
print(f'\nDataset limpio: {len(df_limpio):,} posts organicos')


In [ ]:
# Visualizacion: Fake Engagement
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('PASO 1: Fake Engagement - Deteccion por regla matematica',
             fontsize=13, fontweight='bold')

fake_df = df[df['es_fake']==1]
real_df = df[df['es_fake']==0]

ax = axes[0]
bars = ax.bar(['Organico', 'Fake'], [len(real_df), len(fake_df)],
               color=[BLUE, CORAL], width=0.5)
ax.bar_label(bars, fmt='%d', fontsize=11, fontweight='bold')
ax.set_title('Volumen: Organico vs Fake')
ax.set_ylabel('N posts')

ax = axes[1]
pct_plat = (df.groupby('platform')['es_fake'].mean()*100).sort_values()
colores_p = [CORAL if v == pct_plat.max() else BLUE for v in pct_plat.values]
b2 = ax.barh(pct_plat.index, pct_plat.values, color=colores_p, height=0.55)
for bar, v in zip(b2, pct_plat.values):
    ax.text(v+0.05, bar.get_y()+bar.get_height()/2,
            f'{v:.1f}%', va='center', color=WHITE, fontsize=9, fontweight='bold')
ax.set_title('% Fake por plataforma')
ax.set_xlabel('% posts con fake engagement')

ax = axes[2]
muestra = real_df.sample(800, random_state=SEED)
ax.scatter(np.log1p(muestra['impressions']), np.log1p(muestra['engagement_rate']),
           c=BLUE, s=10, alpha=0.4, label='Organico (muestra)')
ax.scatter(np.log1p(fake_df['impressions']), np.log1p(fake_df['engagement_rate']),
           c=CORAL, s=30, alpha=0.8, marker='X', label=f'Fake ({len(fake_df)})')
ax.legend(fontsize=9)
ax.set_title('Impresiones vs Engagement (X = fake)')
ax.set_xlabel('log(Impresiones)'); ax.set_ylabel('log(Engagement Rate)')

plt.tight_layout(); plt.show()
print('Los posts fake tienen engagement_rate imposible organicamente.')
print('Se separan ANTES de modelar para no contaminar el aprendizaje.')


---
## PASO 2 - Variables sin leakage y coherencia del sentimiento

### Conexion con Adolfo

Adolfo clasifica el tono: Positivo / Negativo / Neutral con F1=0.937.
Yo agrego: ese tono es coherente con la agresividad del texto?

coherencia_sentimiento usa sentiment_label (Adolfo) y toxicity_score (dataset).
Sin el trabajo de Adolfo, esta variable no existe.

### Los 4 casos de coherencia

| Tono | Toxicidad | Resultado | Ejemplo |
|------|-----------|-----------|---------|
| Positivo | Baja | COHERENTE | "Me encanta, lo recomiendo" |
| Positivo | Alta | INCOHERENTE | "Me ENCANTA!!! que IMBECILES los que no lo usan" |
| Negativo | Alta | COHERENTE | "Pesimo servicio, una verguenza" |
| Negativo | Baja | INCOHERENTE | "Creo que quizas podria mejorar un poco..." |


In [ ]:
# Features sin leakage
FEATURES_CLF = [
    'toxicity_score',           # agresividad del texto
    'intensidad_sentimiento',   # magnitud del sentimiento
    'n_hashtags',               # cantidad de hashtags
    'tiene_mencion',            # si etiqueta a alguien
    'es_fin_semana',            # contexto temporal
    'es_hora_laboral',          # contexto temporal
    'user_past_sentiment_avg',  # historial sentimiento del usuario
    'user_engagement_growth',   # tendencia crecimiento del usuario
    'buzz_change_rate',         # velocidad del buzz
]
# sentiment_score EXCLUIDO: leakage trivial con coherencia_sentimiento

FEATURES_REG = [
    'sentiment_score', 'toxicity_score', 'coherencia_sentimiento',
    'intensidad_sentimiento', 'n_hashtags', 'tiene_mencion',
    'es_fin_semana', 'es_hora_laboral',
    'user_past_sentiment_avg', 'user_engagement_growth', 'buzz_change_rate'
]
# Excluidas por leakage con engagement_rate:
# likes_count, shares_count, comments_count, impressions

X_clf = StandardScaler().fit_transform(df_limpio[FEATURES_CLF].fillna(0))
X_reg = StandardScaler().fit_transform(df_limpio[FEATURES_REG].fillna(0))
y_clf = df_limpio['coherencia_sentimiento']
y_reg = df_limpio['log_engagement']

print(f'Features clasificacion: {len(FEATURES_CLF)}')
print(f'Features regresion:     {len(FEATURES_REG)}')
print(f'Clases: {dict(y_clf.value_counts())}')
print(f'Desbalance: {y_clf.mean()*100:.1f}% coherentes vs {(1-y_clf.mean())*100:.1f}% incoherentes')


In [ ]:
# Visualizacion: Los 4 cuadrantes de coherencia
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('PASO 2: Coherencia del Sentimiento - Los 4 cuadrantes',
             fontsize=13, fontweight='bold')

ax = axes[0]
coh1 = df_limpio[df_limpio['coherencia_sentimiento']==1].sample(1200, random_state=SEED)
coh0 = df_limpio[df_limpio['coherencia_sentimiento']==0].sample(800, random_state=SEED)
n_coh = len(df_limpio[df_limpio['coherencia_sentimiento']==1])
n_inc = len(df_limpio[df_limpio['coherencia_sentimiento']==0])

ax.scatter(coh0['toxicity_score'], coh0['sentiment_score'],
           c=CORAL, s=12, alpha=0.45, label=f'Incoherente (n={n_inc:,})')
ax.scatter(coh1['toxicity_score'], coh1['sentiment_score'],
           c=GREEN, s=12, alpha=0.45, label=f'Coherente (n={n_coh:,})')
ax.axhline(0,   color=MUTED, linewidth=1.2, linestyle='--', alpha=0.6)
ax.axvline(0.5, color=GOLD,  linewidth=1.5, linestyle='--', alpha=0.7, label='umbral tox=0.5')
for x, y, txt, c in [
    (0.12, 0.88, 'COHERENTE
Pos + tox baja',    GREEN),
    (0.62, 0.88, 'INCOHERENTE
Pos + tox alta',  CORAL),
    (0.12, 0.05, 'INCOHERENTE
Neg + tox baja',  CORAL),
    (0.62, 0.05, 'COHERENTE
Neg + tox alta',    GREEN),
]:
    ax.text(x, y, txt, transform=ax.transAxes, color=c,
            fontsize=8.5, fontweight='bold', alpha=0.9)
ax.set_xlabel('Toxicidad'); ax.set_ylabel('Sentimiento')
ax.set_title('Scatter: cada punto es un post
Color = coherencia real de la formula')
ax.legend(fontsize=8.5, markerscale=3)

ax = axes[1]
coh_sent = df_limpio.groupby('sentiment_label')['coherencia_sentimiento'].mean()
order = ['Negative','Neutral','Positive']
vals  = [coh_sent[l] for l in order]
bars  = ax.bar(order, vals, color=[CORAL, GOLD, GREEN], width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01,
            f'{val:.2f}', ha='center', color=WHITE, fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.2)
ax.set_title('Coherencia por tono (clasificado por Adolfo)')
ax.set_ylabel('Coherencia promedio')

plt.tight_layout(); plt.show()
print('Neutral = 1.00 siempre coherente por definicion.')
print('Positivo = 0.49: la mitad escribe su positividad con agresividad.')


---
## PASO 3 - No supervisado: PCA

**Pregunta:** Tienen estructura las 9 variables de contexto sin leakage?

Cuanta varianza explican nos dice si las senales de contexto son ricas o pobres.

In [ ]:
# PCA
pca = PCA(n_components=len(FEATURES_CLF), random_state=SEED)
pca.fit(X_clf)
varianza_ind  = pca.explained_variance_ratio_
varianza_acum = np.cumsum(varianza_ind)

print('PCA sobre 9 variables de contexto (sin leakage):')
for n, v in zip([2,3,5,7,9], varianza_acum[[1,2,4,6,8]]):
    print(f'  {n} componentes: {v*100:.1f}% varianza acumulada')

print()
print('Comparacion honesta:')
print(f'  Variables de contexto sin leakage: 5 comp = {varianza_acum[4]*100:.1f}%')
print(f'  Con likes/shares/impressions:       5 comp ~ 85%')
print(f'  La diferencia ({85-varianza_acum[4]*100:.1f} pts) es exactamente el leakage.')
print()
print('El engagement NO esta en el contexto del post.')
print('Esta en como reacciona la audiencia, algo que no existe antes de publicar.')

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(1, len(FEATURES_CLF)+1), varianza_ind*100,
       color=BLUE, alpha=0.8, label='Varianza individual')
ax.plot(range(1, len(FEATURES_CLF)+1), varianza_acum*100,
        'o-', color=GOLD, linewidth=2, markersize=7, label='Varianza acumulada')
ax.axhline(80, linestyle='--', color=GREEN, alpha=0.7, linewidth=1.5, label='80% referencia')
ax.set_xticks(range(1, len(FEATURES_CLF)+1))
ax.set_xlabel('Componente principal')
ax.set_ylabel('Varianza explicada (%)')
ax.set_title('Scree Plot PCA - Variables de contexto sin leakage')
ax.legend()
plt.tight_layout(); plt.show()


---
## PASO 4 - No supervisado: K-Means

**Pregunta:** Existen grupos naturales de posts organicos segun su contexto?

Evaluamos k optimo con Silhouette Score.

In [ ]:
# K-Means
X_pca5 = PCA(n_components=5, random_state=SEED).fit_transform(X_clf)
X_pca2 = PCA(n_components=2, random_state=SEED).fit_transform(X_clf)

print('Silhouette Score por k:')
sils = []
for k in range(2, 7):
    km  = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    lbl = km.fit_predict(X_pca5)
    s   = silhouette_score(X_pca5, lbl, sample_size=2000, random_state=SEED)
    sils.append((k, round(s, 4)))
    print(f'  k={k}: {s:.4f}')

best_k = max(sils, key=lambda x: x[1])[0]
print(f'\nk optimo: {best_k}')

km_final = KMeans(n_clusters=best_k, random_state=SEED, n_init=10)
df_limpio = df_limpio.copy()
df_limpio['cluster'] = km_final.fit_predict(X_pca5)

print('\nPerfil de clusters:')
perfil = df_limpio.groupby('cluster')[[
    'coherencia_sentimiento','toxicity_score','sentiment_score','log_engagement'
]].mean().round(4)
print(perfil.to_string())


In [ ]:
# Visualizacion K-Means
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('PASO 4: K-Means - Grupos naturales de posts organicos',
             fontsize=13, fontweight='bold')

COLORES_K = [GREEN, CORAL, BLUE, GOLD]
clusters_arr = df_limpio['cluster'].values

ax = axes[0]
for i in range(best_k):
    mask  = clusters_arr == i
    coh_m = df_limpio[mask]['coherencia_sentimiento'].mean()
    tipo  = 'Coherente' if coh_m > 0.7 else 'Incoherente'
    ax.scatter(X_pca2[mask,0], X_pca2[mask,1], c=COLORES_K[i],
               s=8, alpha=0.4, label=f'C{i}: coh={coh_m:.2f} ({tipo})')
ax.legend(fontsize=9, markerscale=4)
ax.set_title('PCA 2D coloreado por cluster')
ax.set_xlabel('Componente 1'); ax.set_ylabel('Componente 2')

ax = axes[1]
n_coh2 = len(df_limpio[df_limpio['coherencia_sentimiento']==1])
n_inc2 = len(df_limpio[df_limpio['coherencia_sentimiento']==0])
total2 = n_coh2 + n_inc2
pct_c  = n_coh2/total2*100
pct_i  = n_inc2/total2*100

ax.barh(['Posts organicos'], [pct_c], color=GREEN, height=0.45)
ax.barh(['Posts organicos'], [pct_i], left=[pct_c], color=CORAL, height=0.45)
ax.text(pct_c/2, 0, f'{pct_c:.1f}%
Coherente
({n_coh2:,})',
        ha='center', va='center', color=WHITE, fontsize=12, fontweight='bold')
ax.text(pct_c+pct_i/2, 0, f'{pct_i:.1f}%
Incoherente
({n_inc2:,})',
        ha='center', va='center', color=WHITE, fontsize=12, fontweight='bold')
ax.set_xlim(0, 100)
ax.set_title('Que separo K-Means?
Coherente vs Incoherente')
ax.set_xlabel('% de posts organicos')

plt.tight_layout(); plt.show()
print(f'2 de cada 3 posts son coherentes ({pct_c:.1f}%).')
print(f'1 de cada 3 es incoherente ({pct_i:.1f}%): senal que el creador puede corregir.')


---
## PASO 5 - Trade-off de Timing: Impresiones vs Engagement

**Insight:** El horario con mas impresiones no es el horario con mas engagement.
Publicar cuando mas gente te ve no es cuando mas gente reacciona.

Esto es un **trade-off**: mejorar uno implica sacrificar el otro.

In [ ]:
# Trade-off timing
hora_stats = df_limpio.groupby('hora').agg(
    imp_media  = ('impressions',     'mean'),
    eng_media  = ('engagement_rate', 'mean'),
    n_posts    = ('impressions',     'count')
).reset_index()

umbral_imp = hora_stats['imp_media'].mean()
umbral_eng = hora_stats['eng_media'].mean()

hora_stats['zona'] = np.where(
    (hora_stats['imp_media'] > umbral_imp) & (hora_stats['eng_media'] > umbral_eng),
    'sweet_spot',
    np.where(hora_stats['imp_media'] > umbral_imp, 'alto_alcance',
    np.where(hora_stats['eng_media'] > umbral_eng, 'alto_engagement', 'bajo'))
)

print('TOP 5 horas por IMPRESIONES:')
print(hora_stats.nlargest(5,'imp_media')[['hora','imp_media','eng_media','zona']].to_string(index=False))

print('\nTOP 5 horas por ENGAGEMENT:')
print(hora_stats.nlargest(5,'eng_media')[['hora','eng_media','imp_media','zona']].to_string(index=False))

print('\nPor dia de semana:')
dia_stats = df_limpio.groupby('dia_semana').agg(
    imp_media=('impressions','mean'),
    eng_media=('engagement_rate','mean')
).reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
print(dia_stats.sort_values('imp_media',ascending=False).round(2).to_string())

combo = df_limpio.groupby(['dia_semana','hora'])['impressions'].mean().reset_index()
print('\nTop 5 combinaciones dia+hora (impresiones):')
print(combo.nlargest(5,'impressions').to_string(index=False))


In [ ]:
# Visualizacion trade-off
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('PASO 5: Trade-off Timing - Impresiones vs Engagement por hora',
             fontsize=13, fontweight='bold')

from matplotlib.patches import Patch

COLORES_ZONA = {'sweet_spot':GREEN,'alto_alcance':BLUE,'alto_engagement':GOLD,'bajo':MUTED}

ax = axes[0]
for _, row in hora_stats.iterrows():
    color = COLORES_ZONA[row['zona']]
    ax.scatter(row['imp_media'], row['eng_media'], c=color, s=130, zorder=3)
    ax.annotate(f"{int(row['hora'])}h", (row['imp_media'], row['eng_media']),
                textcoords='offset points', xytext=(5,5), fontsize=7.5, color=color)
ax.axvline(umbral_imp, color=MUTED, linestyle='--', alpha=0.5)
ax.axhline(umbral_eng, color=MUTED, linestyle='--', alpha=0.5)
leyenda = [Patch(color=GREEN, label='Sweet spot'),
           Patch(color=BLUE,  label='Alto alcance'),
           Patch(color=GOLD,  label='Alto engagement'),
           Patch(color=MUTED, label='Bajo en ambas')]
ax.legend(handles=leyenda, fontsize=8.5)
ax.set_xlabel('Impresiones promedio')
ax.set_ylabel('Engagement Rate promedio')
ax.set_title('Trade-off por hora del dia
cada punto = una hora del dia')

ax2 = axes[1]
ax2.set_facecolor('#21262D')
horas = hora_stats['hora'].values
ax2.plot(horas, hora_stats['imp_media']/1000, 'o-', color=BLUE,
         linewidth=2, markersize=6, label='Impresiones (miles)')
ax2r = ax2.twinx()
ax2r.set_facecolor('#21262D')
ax2r.plot(horas, hora_stats['eng_media'], 's--', color=CORAL,
          linewidth=2, markersize=6, label='Engagement Rate')
ax2.set_xlabel('Hora del dia')
ax2.set_ylabel('Impresiones (miles)', color=BLUE)
ax2r.set_ylabel('Engagement Rate', color=CORAL)
ax2.tick_params(axis='y', colors=BLUE)
ax2r.tick_params(axis='y', colors=CORAL)
ax2.set_title('Curvas por hora
cuando una sube la otra no necesariamente sube')
lines1, labs1 = ax2.get_legend_handles_labels()
lines2, labs2 = ax2r.get_legend_handles_labels()
ax2.legend(lines1+lines2, labs1+labs2, fontsize=8.5)
ax2r.spines[:].set_color('#2D333B')

plt.tight_layout(); plt.show()

hora_max_imp = hora_stats.loc[hora_stats['imp_media'].idxmax()]
hora_max_eng = hora_stats.loc[hora_stats['eng_media'].idxmax()]
print(f'Max impresiones: {int(hora_max_imp["hora"])}:00h ({hora_max_imp["imp_media"]:.0f} imp)')
print(f'Max engagement:  {int(hora_max_eng["hora"])}:00h (rate={hora_max_eng["eng_media"]:.4f})')
print('No coinciden. El creador decide que priorizar.')
print('Alcance: Sabado 19:00 (peak combinado 63.687 imp)')
print('Reaccion: madrugada 01:00-04:00')


---
## PASO 6 - Supervisado: Clasificacion

**Target:** coherencia_sentimiento (0=incoherente, 1=coherente)  
**Features:** 9 variables de contexto sin leakage  
**Metrica:** F1-macro - clases desbalanceadas (67% vs 33%)

**Por que F1-macro y no Accuracy?**  
Un modelo que predijera siempre coherente tendria 66.9% de accuracy sin aprender nada.  
F1-macro equilibra el rendimiento en ambas clases sin importar su tamano.


In [ ]:
# Clasificacion
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelos_clf = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED,
                                               class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=SEED,
                                                   class_weight='balanced', n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    'SVM':                 SVC(class_weight='balanced', random_state=SEED),
}

print('CLASIFICACION: coherencia_sentimiento - F1-macro (5-fold CV)')
print()
resultados_clf = {}
for nombre, clf in modelos_clf.items():
    t0     = time.time()
    f1s    = cross_val_score(clf, X_clf, y_clf, cv=skf, scoring='f1_macro',  n_jobs=-1)
    accs   = cross_val_score(clf, X_clf, y_clf, cv=skf, scoring='accuracy',  n_jobs=-1)
    resultados_clf[nombre] = {
        'f1_mean':  round(f1s.mean(),4),
        'f1_std':   round(f1s.std(),4),
        'acc_mean': round(accs.mean(),4),
        'tiempo':   round(time.time()-t0,1)
    }
    print(f'{nombre}:')
    print(f'  F1-macro: {f1s.mean():.4f} +/- {f1s.std():.4f}')
    print(f'  Accuracy: {accs.mean():.4f}  |  Tiempo: {resultados_clf[nombre]["tiempo"]}s')
    print()

mejor_clf = max(resultados_clf, key=lambda x: resultados_clf[x]['f1_mean'])
print(f'Mejor modelo: {mejor_clf} (F1={resultados_clf[mejor_clf]["f1_mean"]:.4f})')


In [ ]:
# Visualizacion clasificacion
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('PASO 6: Clasificacion - F1-macro vs Accuracy',
             fontsize=13, fontweight='bold')

nombres = list(resultados_clf.keys())
f1s     = [resultados_clf[n]['f1_mean'] for n in nombres]
stds    = [resultados_clf[n]['f1_std']  for n in nombres]
accs    = [resultados_clf[n]['acc_mean']for n in nombres]
colores_b = [GREEN if n == mejor_clf else BLUE for n in nombres]

ax = axes[0]
barras = ax.bar(nombres, f1s, yerr=stds, color=colores_b,
                capsize=6, error_kw={'color':WHITE,'linewidth':2}, width=0.55)
ax.bar_label(barras, fmt='%.4f', fontsize=10, fontweight='bold', padding=5)
ax.set_ylim(0.4, 0.8)
ax.set_title('F1-macro por modelo (5-fold CV)')
ax.set_ylabel('F1-macro')
ax.tick_params(axis='x', rotation=15)

ax = axes[1]
x_pos = np.arange(len(nombres)); w = 0.35
b1 = ax.bar(x_pos-w/2, f1s,  w, color=GREEN, alpha=0.85, label='F1-macro')
b2 = ax.bar(x_pos+w/2, accs, w, color=CORAL, alpha=0.85, label='Accuracy')
ax.bar_label(b1, fmt='%.3f', fontsize=8.5, padding=3)
ax.bar_label(b2, fmt='%.3f', fontsize=8.5, padding=3)
ax.set_xticks(x_pos); ax.set_xticklabels(nombres, rotation=15)
ax.set_title('F1-macro vs Accuracy
Accuracy engana con clases desbalanceadas')
ax.legend(); ax.set_ylim(0.4, 0.85)

plt.tight_layout(); plt.show()
print('F1 honesto ~0.65: predecimos coherencia sin el texto, solo con senales indirectas.')
print('Con TF-IDF (como Adolfo), el F1 subiria a ~0.85+.')


---
## PASO 7 - Supervisado: Regresion

**Target:** log_engagement (sin leakage)  
**Pregunta:** Puede predecirse el engagement desde variables de contexto puro?

**Resultado esperado:** R2 cercano a 0, y ese ES el hallazgo mas importante.

- R2 = 0: el modelo predice igual que decir siempre el promedio
- R2 < 0: el modelo es peor que predecir el promedio
- R2 = 0.99 con leakage: trampa documentada


In [ ]:
# Regresion
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

modelos_reg = {
    'Ridge':        Ridge(),
    'RF Regressor': RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
}

print('REGRESION: log_engagement - RMSE + R2 (5-fold CV)')
print()
resultados_reg = {}
for nombre, reg in modelos_reg.items():
    rmse = np.sqrt(-cross_val_score(reg, X_reg, y_reg, cv=kf,
                   scoring='neg_mean_squared_error', n_jobs=-1))
    r2   = cross_val_score(reg, X_reg, y_reg, cv=kf, scoring='r2', n_jobs=-1)
    resultados_reg[nombre] = {
        'rmse_mean': round(rmse.mean(),4), 'r2_mean': round(r2.mean(),4)
    }
    print(f'{nombre}:')
    print(f'  RMSE: {rmse.mean():.4f} +/- {rmse.std():.4f}')
    print(f'  R2:   {r2.mean():.4f} +/- {r2.std():.4f}')
    print()

print('INTERPRETACION:')
print('El engagement NO es predecible desde el contexto puro.')
print('Depende de cuantas personas vieron el post y como reaccionaron.')
print('Esas variables NO existen antes de publicar.')
print()
print('Con leakage (likes, shares, impressions como features):')
print('  R2 seria ~0.99 - trampa documentada, el modelo aprende la formula')
print('  no el fenomeno real.')


In [ ]:
# Visualizacion regresion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PASO 7: Regresion - El engagement no es predecible desde el contexto',
             fontsize=13, fontweight='bold')

ax = axes[0]
modelos_comp = {
    'Ridge
(sin leakage)':    resultados_reg['Ridge']['r2_mean'],
    'RF
(sin leakage)':       resultados_reg['RF Regressor']['r2_mean'],
    'RF
(CON leakage)
trampa': 0.9907,
}
cols_comp = [GREEN, GREEN, CORAL]
barras_r2 = ax.bar(list(modelos_comp.keys()), list(modelos_comp.values()),
                    color=cols_comp, width=0.5)
ax.bar_label(barras_r2, fmt='%.4f', fontsize=10, fontweight='bold', padding=4)
ax.axhline(0, color=WHITE, linewidth=1.5, linestyle='--', alpha=0.5)
ax.set_title('R2 con y sin leakage
la diferencia ES el leakage')
ax.set_ylabel('R2')

ax = axes[1]
X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=SEED)
ridge_fit = Ridge().fit(X_tr, y_tr)
y_pred    = ridge_fit.predict(X_te)
ax.scatter(y_te, y_pred, c=BLUE, s=8, alpha=0.3)
ax.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()],
        color=CORAL, linewidth=2, linestyle='--', label='Prediccion perfecta')
ax.set_xlabel('Engagement real (log)')
ax.set_ylabel('Engagement predicho (log)')
ax.set_title('Predicho vs Real (Ridge)
Si fuera perfecto, todos en la linea')
ax.legend()

plt.tight_layout(); plt.show()


---
## PASO 8 - Optimizacion de Hiperparametros

**Modelo:** Gradient Boosting (mejor clasificador F1=0.6457)  
**Metodos:** GridSearchCV + RandomizedSearchCV

**Trade-off entre metodos:**
- GridSearchCV: exhaustivo dentro de la rejilla, garantiza el mejor del grid
- RandomizedSearchCV: explora espacio mas amplio, eficiente para espacios grandes


In [ ]:
# Optimizacion
base_gb = GradientBoostingClassifier(n_estimators=100, random_state=SEED)
base_f1 = cross_val_score(base_gb, X_clf, y_clf, cv=skf,
                           scoring='f1_macro', n_jobs=-1).mean()
print(f'Base Gradient Boosting F1-macro: {base_f1:.4f}')
print()

# GridSearchCV
param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.1],
}
n_combos = 2*2*2
print(f'GridSearchCV: {n_combos} combinaciones x 3 folds = {n_combos*3} entrenamientos')
grid = GridSearchCV(GradientBoostingClassifier(random_state=SEED),
                    param_grid, cv=3, scoring='f1_macro', n_jobs=-1)
grid.fit(X_clf, y_clf)
print(f'  F1: {grid.best_score_:.4f} | params: {grid.best_params_}')
print()

# RandomizedSearchCV
print('RandomizedSearchCV: 10 iteraciones x 3 folds = 30 entrenamientos')
rand = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=SEED),
    {'n_estimators': randint(100,300), 'max_depth':[3,4,5],
     'learning_rate':[0.05,0.1,0.15]},
    n_iter=10, cv=3, scoring='f1_macro', random_state=SEED, n_jobs=-1
)
rand.fit(X_clf, y_clf)
print(f'  F1: {rand.best_score_:.4f} | params: {rand.best_params_}')
print()

tabla_opt = pd.DataFrame({
    'Metodo':   ['Base GB', 'GridSearchCV', 'RandomizedSearchCV'],
    'F1-macro': [round(base_f1,4), round(grid.best_score_,4), round(rand.best_score_,4)],
    'Delta':    [0, round(grid.best_score_-base_f1,4), round(rand.best_score_-base_f1,4)]
})
print(tabla_opt.to_string(index=False))


In [ ]:
# Visualizacion optimizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PASO 8: Optimizacion - GridSearchCV vs RandomizedSearchCV',
             fontsize=13, fontweight='bold')

ax = axes[0]
metodos  = ['Base GB', 'GridSearchCV', 'RandomizedSearchCV']
f1_vals  = [base_f1, grid.best_score_, rand.best_score_]
cols_opt = [MUTED, BLUE, GREEN]
b_opt    = ax.bar(metodos, f1_vals, color=cols_opt, width=0.5)
ax.bar_label(b_opt, fmt='%.4f', fontsize=11, fontweight='bold', padding=4)
ax.set_ylim(0.55, 0.75)
ax.set_title('F1-macro: Base vs Optimizado')
ax.set_ylabel('F1-macro')

ax = axes[1]
grid_res = pd.DataFrame(grid.cv_results_)
grid_res['lr_str'] = grid_res['param_learning_rate'].astype(str)
for lr in grid_res['lr_str'].unique():
    sub = grid_res[grid_res['lr_str']==lr].sort_values('param_n_estimators')
    ax.plot(sub['param_n_estimators'], sub['mean_test_score'],
            'o-', linewidth=2, markersize=8, label=f'lr={lr}')
ax.set_xlabel('n_estimators')
ax.set_ylabel('F1-macro CV')
ax.set_title('Sensibilidad del GridSearch
por learning_rate')
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()
print('Si la optimizacion no mejora significativamente:')
print('el cuello de botella son las FEATURES, no los hiperparametros.')


---
## Conclusion del Acto 2

In [ ]:
print('=' * 65)
print('CONCLUSION - ACTO 2: SEPARAR LO REAL DE LO FALSO')
print('=' * 65)

resumen = [
    ('PASO 1 - Fake Engagement',
     '487 posts (4.1%) con rate > 1: imposible organicamente.\n'
     'Separados ANTES de modelar.'),
    ('PASO 2 - Variables sin leakage',
     'coherencia_sentimiento usa el tono de Adolfo + toxicidad.\n'
     'Sin Adolfo, esta variable no existe.'),
    ('PASO 3 - PCA',
     '9 variables de contexto: 5 comp = 48.9% varianza.\n'
     'Con leakage seria ~85%. La diferencia ES el leakage.'),
    ('PASO 4 - K-Means',
     'k=2 optimo: 66.9% coherentes vs 33.1% incoherentes.'),
    ('PASO 5 - Trade-off timing',
     'Max impresiones: Sabado 19:00 (63.687 imp).\n'
     'Max engagement:  01:00h (rate 0.1506). No coinciden.'),
    ('PASO 6 - Clasificacion (lo que SI podemos predecir)',
     f'Gradient Boosting: F1-macro = {resultados_clf["Gradient Boosting"]["f1_mean"]:.4f}\n'
     f'Con optimizacion: F1 = {grid.best_score_:.4f}'),
    ('PASO 7 - Regresion (lo que NO podemos predecir)',
     f'Ridge R2 = {resultados_reg["Ridge"]["r2_mean"]:.4f}\n'
     'El engagement depende de la audiencia, no del creador.'),
    ('PASO 8 - Optimizacion',
     'GridSearch y RandomizedSearch: las features son la fuente de valor.'),
]

for titulo, cuerpo in resumen:
    print(f'\n{titulo}:')
    for linea in cuerpo.split('\n'):
        print(f'  {linea}')

print()
print('=' * 65)
print('SENALES QUE EL CREADOR PUEDE CONTROLAR:')
print('  El tono del texto      (Adolfo, F1=0.937)')
print('  La coherencia          (este analisis, F1=0.65)')
print('  El timing de publicacion (Sabado 19:00 para alcance)')
print()
print('SENALES QUE NO PUEDE CONTROLAR:')
print('  Cuantas personas lo veran  (impressions -> algoritmo)')
print('  Cuantos daran like/share   (depende de la audiencia)')
print('  El engagement_rate final   (R2~0 sin leakage)')
print('=' * 65)
print()
print('La diferencia entre un insight util y uno inutil es si')
print('puedes actuar sobre el ANTES de publicar.')
